In [1]:
import pandas as pd
import numpy as np
import os, re
from pathlib import Path

def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "PROJECT_EXPERIMENT_GUIDE.md").exists() and (candidate / "README.md").exists():
            return candidate
    raise RuntimeError("Could not find the XLZD repo root from the current working directory.")


REPO_ROOT = find_repo_root()

In [2]:
# Grab all files, radii, and heights
validation = True

sim_path = REPO_ROOT.parent / "sim"
if validation:
    sim_data_path = sim_path / "simulation_data"
    data_path = REPO_ROOT / "data" / "val_data"
else:
    sim_data_path = sim_path / "simulation_data_train"
    data_path = REPO_ROOT / "data" / "train_data"
entries = []
for file in sim_data_path.glob("R*H*F*.csv"):
    pattern = r"R(\d+)_H(\d+)_F(\d+)"
    match = re.search(pattern, str(file))

    if match:
        radius = match.group(1)
        height = match.group(2)
        fidelity = match.group(3)
        entries.append((file, int(radius), int(height), fidelity))

In [3]:
# Cut down the file
for entry in entries:
    file = entry[0]
    radius = entry[1]
    height = entry[2]
    fidelity = entry[3]
    
    data = pd.read_csv(file)
    data = data.copy()
    len_before = len(data)

    print("="*50)
    
    # If not zero-centered, make zero centered
    if (min(data["sz"]) < -15):
        print(f"Minimum z of {min(data["sz"])}, adding {height} to z and sz")
        data["z"] = data["z"] + height
        data["sz"] = data["sz"] + height
    
    # Calculate radius and cut based on radius and height
    data.drop(data[np.sqrt(data["x"]**2 + data["y"]**2) > radius].index, inplace=True)
    data.drop(data[data["z"] > 2*height].index, inplace=True)
    data.drop(data[data["z"] < 0].index, inplace=True)
    #data.drop(data[data["ETPC"] > 3].index, inplace=True)
    data.drop(data[data["ETPC"] < 0.5].index, inplace=True)
    len_after = len(data)
    
    # Save the new dataframe
    new_file = data_path / f"{file.stem}_withcuts{file.suffix}"
    if os.path.exists(new_file):
        os.remove(new_file)
    data.to_csv(new_file, index=False)
    print(f"Dropped {round(((len_before - len_after) / len_before)*100, 2)}% events")
    print(f"Kept {len_after} events")
    print(f"File saved to {new_file}")
    print()

    # Add to file_manifest
    if validation:
        manifest_path = data_path / "validation_manifest.csv"
    else:
        manifest_path = data_path / "file_manifest.csv"
    manifest = pd.read_csv(manifest_path)
    if new_file.name in manifest["filename"]:
        print(f"{new_file} already in manifest, passing")
        pass
    else:
        manifest.loc[len(manifest)] = [new_file.name, radius, height, height, fidelity]
        print(f"Wrote file {new_file.name} to manifest with radius {radius}, height {height}, z_center {height}, and fidelity {fidelity}")
    manifest.to_csv(manifest_path, index=False)

Dropped 0.0% events
Kept 207872 events
File saved to /home/pknauss/XLZD/data/val_data/R1712_H825_F1_2447keVgamma_withcuts.csv

Wrote file R1712_H825_F1_2447keVgamma_withcuts.csv to manifest with radius 1712, height 825, z_center 825, and fidelity 1
Dropped 0.0% events
Kept 201307 events
File saved to /home/pknauss/XLZD/data/val_data/R857_H1767_F1_2447keVgamma_withcuts.csv

Wrote file R857_H1767_F1_2447keVgamma_withcuts.csv to manifest with radius 857, height 1767, z_center 1767, and fidelity 1
Dropped 0.0% events
Kept 205653 events
File saved to /home/pknauss/XLZD/data/val_data/R1246_H1381_F1_2447keVgamma_withcuts.csv

Wrote file R1246_H1381_F1_2447keVgamma_withcuts.csv to manifest with radius 1246, height 1381, z_center 1381, and fidelity 1
Dropped 0.0% events
Kept 166840 events
File saved to /home/pknauss/XLZD/data/val_data/R400_H200_F1_2447keVgamma_withcuts.csv

Wrote file R400_H200_F1_2447keVgamma_withcuts.csv to manifest with radius 400, height 200, z_center 200, and fidelity 1
Dr